# Phase 5b — Zero-Shot Image-to-Text Similarity
The trained trimodal fusion classifier hit 99% accuracy by shortcutting through the fixed per-class text embedding (label leakage), not by learning genuine multimodal signal. This notebook replaces that with EyeCLIP's actual intended zero-shot mechanism: **no trainable classifier at all.**

**Method:** classify each image by which of the 8 class text-description embeddings its own image embedding is most similar to (cosine similarity). No fusion layer, no training, no way for the label to leak in through a shortcut -- the model has to genuinely relate the fundus image to the clinical description text.

This is a completely different question from Phase 1-5a: it does not use train-set images at all as "support" -- every test image is compared only to the 8 fixed text descriptions.

In [1]:
import pathlib
clip_path = pathlib.Path(r"E:\rop\codes\EyeCLIP\eyeclip\clip.py")  # your actual path
text = clip_path.read_text()
text = text.replace("from pkg_resources import packaging", "import packaging")
clip_path.write_text(text)

9800

In [2]:
import sys
sys.path.insert(0, r"E:\rop\codes\EyeCLIP")  # CHANGE to your actual path

import torch, eyeclip
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, f1_score, roc_auc_score, confusion_matrix

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
CKPT_PATH = r"E:\rop\codes\eyeclip.pt"           # CHANGE to actual path -- same checkpoint as Phase 0/1
DISEASE_DESC_PATH = r"E:\rop\dataset\ROP-VL\disease_description.xlsx"
EMB_PATH = r"E:\rop\codes\eyeclip_embeddings.zip"
FUSED_EMB_PATH = r"E:\rop\codes\fused_embeddings.npz"
SPLIT_DIR = Path(r"E:\rop\dataset\ROP-VL\split")  # CHANGE to actual path

model, preprocess = eyeclip.load("ViT-B/32", device=DEVICE, jit=False)
ckpt = torch.load(CKPT_PATH, map_location=DEVICE)
state = ckpt.get("model_state_dict") or ckpt.get("state_dict") or ckpt
model.load_state_dict(state, strict=False)
model.eval()
for p in model.parameters():
    p.requires_grad = False
print("EyeCLIP loaded on", DEVICE)

e:\.venv\Lib\site-packages\torch\jit\_serialization.py:170: FutureWarning: `torch.jit.load` is not supported in Python 3.14+ and may break. Please switch to `torch.export`.
  warnings.warn(


EyeCLIP loaded on cpu


In [3]:
desc_df = pd.read_excel(DISEASE_DESC_PATH)
assert hasattr(eyeclip, "tokenize"), "eyeclip module has no tokenize() -- check the repo's text API"
tokens = eyeclip.tokenize(desc_df["description"].tolist(), truncate=True).to(DEVICE)

with torch.no_grad():
    text_embeddings = model.encode_text(tokens).float().cpu().numpy()  # [8, 512]

classes_sorted = desc_df["disease"].tolist()
print("classes (in text-embedding order):", classes_sorted)
print("text_embeddings shape:", text_embeddings.shape)

classes (in text-embedding order): ['normal', 'Stage 1 ROP', 'Stage 2 ROP', 'Stage 3 ROP', 'Stage 4 ROP', 'Stage 5 ROP', 'A-ROP', 'Laser-treated ROP']
text_embeddings shape: (8, 512)


In [4]:
data = np.load(EMB_PATH)
X_test_img, y_test_arr = data["X_test"], data["y_test"]

test_df = pd.read_csv(SPLIT_DIR / "test.csv")
assert len(test_df) == len(X_test_img), "test.csv row count != X_test row count"
y_test_str = test_df["categories"].values  # use string labels directly, avoids trusting Phase 1's label2idx order

missing = set(y_test_str) - set(classes_sorted)
assert not missing, f"test classes with no text description: {missing}"

## Zero-shot classification: cosine similarity, image embedding vs each class's text embedding
This mirrors CLIP's standard zero-shot recipe: normalize both embedding sets, take the dot product (= cosine similarity since both are unit-normalized), pick the class with highest similarity per image.

In [5]:
def zero_shot_predict(image_embs, text_embs, classes):
    """image_embs [N,512], text_embs [C,512] -> preds [N] (class names), sims [N,C]
    Both are L2-normalized before the dot product so it equals cosine similarity.
    """
    img_n = image_embs / (np.linalg.norm(image_embs, axis=1, keepdims=True) + 1e-8)
    txt_n = text_embs  / (np.linalg.norm(text_embs,  axis=1, keepdims=True) + 1e-8)
    sims = img_n @ txt_n.T           # [N, C], higher = more similar
    pred_idx = sims.argmax(axis=1)
    preds = np.array([classes[i] for i in pred_idx])
    return preds, sims

preds, sims = zero_shot_predict(X_test_img, text_embeddings, classes_sorted)

acc = accuracy_score(y_test_str, preds)
precision, recall, f1, support = precision_recall_fscore_support(
    y_test_str, preds, labels=classes_sorted, average=None, zero_division=0)
macro_f1 = f1.mean()

print(f"Zero-shot (no training, no fusion) -- Test accuracy: {acc:.4f}   Macro-F1: {macro_f1:.4f}\n")
print("Reference -- Phase 1 (image-only, trained linear): acc=0.3063  macro_f1=0.2469")
print("Reference -- Phase 3 (image+metadata, trained):    acc=0.3188  macro_f1=0.2174")
print("Reference -- Phase 5a (leaky trimodal, INVALID):    acc=0.9938  macro_f1=0.9621 (do not compare against this)\n")

for c, p, r, f, s in zip(classes_sorted, precision, recall, f1, support):
    print(f"  {c:20s}  P={p:.3f}  R={r:.3f}  F1={f:.3f}  n={s}")

cm = pd.DataFrame(confusion_matrix(y_test_str, preds, labels=classes_sorted),
                   index=classes_sorted, columns=classes_sorted)
print("\nConfusion matrix:")
print(cm)

pd.DataFrame([{"representation": "zero_shot_image_to_text", "accuracy": acc, "macro_f1": macro_f1}]
            ).to_csv("phase5b_zeroshot_results.csv", index=False)
cm.to_csv("phase5b_zeroshot_confusion_matrix.csv")

Zero-shot (no training, no fusion) -- Test accuracy: 0.3031   Macro-F1: 0.0582

Reference -- Phase 1 (image-only, trained linear): acc=0.3063  macro_f1=0.2469
Reference -- Phase 3 (image+metadata, trained):    acc=0.3188  macro_f1=0.2174
Reference -- Phase 5a (leaky trimodal, INVALID):    acc=0.9938  macro_f1=0.9621 (do not compare against this)

  normal                P=0.303  R=1.000  F1=0.465  n=97
  Stage 1 ROP           P=0.000  R=0.000  F1=0.000  n=37
  Stage 2 ROP           P=0.000  R=0.000  F1=0.000  n=52
  Stage 3 ROP           P=0.000  R=0.000  F1=0.000  n=38
  Stage 4 ROP           P=0.000  R=0.000  F1=0.000  n=12
  Stage 5 ROP           P=0.000  R=0.000  F1=0.000  n=5
  A-ROP                 P=0.000  R=0.000  F1=0.000  n=25
  Laser-treated ROP     P=0.000  R=0.000  F1=0.000  n=54

Confusion matrix:
                   normal  Stage 1 ROP  Stage 2 ROP  Stage 3 ROP  Stage 4 ROP  \
normal                 97            0            0            0            0   
Stage 1 ROP    

## Zero-shot open-set: same idea, using only 6 known-class text anchors
Since there's no training here, "support set" and "shots" don't apply -- text descriptions ARE the class anchors, always available in full. This tests: does the image's similarity to its *best-matching known-class description* drop noticeably when the image is actually Stage 4/5 (unknown)?

In [6]:
UNKNOWN_CLASSES = ["Stage 4 ROP", "Stage 5 ROP"]
known_classes = [c for c in classes_sorted if c not in UNKNOWN_CLASSES]
known_idx = [classes_sorted.index(c) for c in known_classes]
known_text_embs = text_embeddings[known_idx]

test_is_known = np.isin(y_test_str, known_classes)

preds_known, sims_known = zero_shot_predict(X_test_img, known_text_embs, known_classes)
best_sim = sims_known.max(axis=1)          # higher = more confident match to some known class
unknown_score = -best_sim                   # lower similarity = more "unknown-like"; higher score = more unknown

try:
    auroc = roc_auc_score(~test_is_known, unknown_score)
except ValueError:
    auroc = float("nan")
print(f"Zero-shot open-set AUROC (known vs unknown separability, no threshold needed): {auroc:.4f}")

# threshold still needed to report Known-F1/Unknown-F1 -- select on a validation split if available,
# otherwise report AUROC as the primary threshold-independent result and use median split as illustration only
median_thresh = np.median(best_sim)
test_accepted = best_sim >= median_thresh
unknown_f1 = f1_score(~test_is_known, ~test_accepted, zero_division=0)
eval_mask = test_is_known & test_accepted
known_f1 = (f1_score(y_test_str[eval_mask], preds_known[eval_mask],
                      labels=known_classes, average="macro", zero_division=0)
            if eval_mask.sum() > 0 else 0.0)
print(f"(illustrative median-split threshold) Known-F1: {known_f1:.4f}  Unknown-F1: {unknown_f1:.4f}")
print("NOTE: for a defensible number, tune this threshold on a held-out val split, not test's own median.")

Zero-shot open-set AUROC (known vs unknown separability, no threshold needed): 0.6154
(illustrative median-split threshold) Known-F1: 0.0720  Unknown-F1: 0.1469
NOTE: for a defensible number, tune this threshold on a held-out val split, not test's own median.


## Interpretation notes
- This result reflects EyeCLIP's actual pretrained image-text alignment quality on ROP images -- with zero task-specific training, so a modest number here is expected and legitimate (compare fairly against Phase 1's ~30%, not Phase 5a's leaked 99%).
- If zero-shot accuracy is well below Phase 1/3, that means EyeCLIP's pretrained text-image alignment doesn't transfer well to this specific clinical vocabulary out of the box -- a real, reportable limitation of using an off-the-shelf checkpoint zero-shot.
- The open-set AUROC here is more trustworthy than Phase 4/4b's, since there is no trained classifier that could have learned a shortcut -- it purely reflects whether Stage 4/5 images are inherently less similar to the 6 known-class descriptions.
- Do not average or directly compare this against Phase 5a's numbers in the final report -- state clearly that Phase 5a was invalidated by label leakage and this zero-shot result is the valid Phase 5 text-branch finding.